In [1]:
!pip install -q -U transformers datasets accelerate evaluate sentencepiece


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 18.5 MB/s eta 0:00:00


In [1]:
import os
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split


<h3>Data preparation</h3>

In [2]:
# --- CONFIGURE PATHS ---
CSV_PATH = "/content/drive/MyDrive/Assignments/Main_Project/python_bugfix_pairs.csv"
CODENET_ROOT = "/content/drive/MyDrive/Assignments/Main_Project/Dataset_Metadata/metadata"
RANDOM_SEED = 42


In [3]:
def read_file_safe(path: str) -> str:
    try:
        with open(path, "r", encoding="utf-8") as f:
            return f.read()
    except Exception:
        try:
            with open(path, "r", encoding="latin-1") as f:
                return f.read()
        except Exception:
            return ""


Load the Buggy & Fixed files, and store its content in the csv file and path

In [4]:
df = pd.read_csv(CSV_PATH)
assert "buggy" in df.columns and "fixed" in df.columns and 'problem_id' in df.columns, "CSV must contain 'buggy' and 'fixed' and 'problem_id' columns"

# --- build full paths and read contents ---
df["buggy_path"] = df.apply(lambda row: os.path.join(CODENET_ROOT, str(row['problem_id']), str(row['buggy'])) + ".csv", axis=1)
df["fixed_path"] = df.apply(lambda row: os.path.join(CODENET_ROOT, str(row['problem_id']), str(row['fixed'])) + ".csv", axis=1)

df["buggy_code"] = df["buggy_path"].apply(read_file_safe)
df["fixed_code"] = df["fixed_path"].apply(read_file_safe)


Delete empty reads

In [ ]:
df = df[(df["buggy_code"].str.strip() != "") & (df["fixed_code"].str.strip() != "")].reset_index(drop=True)
print(f"Paired examples available: {len(df)}")
